# Clasificación de salud fetal con regresión logística

Este notebook muestra un flujo básico de clasificación usando Python.

El objetivo es construir un modelo que prediga la variable `fetal_health` a partir de las demás columnas del archivo `fetal_health.csv`.

Se utilizarán solamente:

- `pandas` para leer y revisar los datos.
- `scikit-learn` para entrenar y evaluar un modelo de **regresión logística**.
- `matplotlib` para visualizar la matriz de confusión.


## 1. Instalación de librerías

Ejecute la siguiente celda solo si su entorno no tiene instaladas las librerías necesarias.


In [ ]:
# Descomente esta línea si necesita instalar las librerías.
# %pip install pandas scikit-learn matplotlib

## 2. Importar librerías

En esta sección se cargan las herramientas que se usarán durante todo el notebook.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 3. Leer el archivo CSV

El archivo `fetal_health.csv` debe estar en la misma carpeta que este notebook.

Si el archivo se encuentra en otra ubicación, cambie la ruta de la variable `CSV_PATH`.


In [ ]:
CSV_PATH = Path("fetal_health.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el archivo fetal_health.csv. "
        "Coloque el archivo en la misma carpeta del notebook o cambie la ruta en CSV_PATH."
    )

df = pd.read_csv(CSV_PATH)

df.head()

## 4. Revisar la base de datos

Antes de entrenar un modelo, es importante mirar la estructura básica de los datos.


In [ ]:
# Número de filas y columnas
print("Tamaño de la base de datos:", df.shape)

# Nombre de las columnas
df.columns

In [ ]:
# Información general de la base de datos
df.info()

In [ ]:
# Revisar si existen datos faltantes
df.isna().sum()

## 5. Revisar la variable objetivo

La columna que se desea predecir es `fetal_health`.

En esta base de datos, las clases suelen estar codificadas así:

- `1`: Normal
- `2`: Sospechoso
- `3`: Patológico


In [ ]:
df["fetal_health"].value_counts().sort_index()

In [ ]:
class_names = {
    1: "Normal",
    2: "Sospechoso",
    3: "Patológico",
}

# Convertimos la clase numérica en una etiqueta de texto.
# Esto hace que los resultados sean más fáciles de leer.
y = df["fetal_health"].astype(int).map(class_names)

y.value_counts()

## 6. Separar características y etiqueta

En un problema de clasificación se separan los datos en dos partes:

- `X`: variables de entrada o características.
- `y`: variable de salida o etiqueta que se desea predecir.


In [ ]:
X = df.drop(columns=["fetal_health"])

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)

## 7. Dividir los datos en entrenamiento y prueba

El modelo aprende con los datos de entrenamiento.

Después, se evalúa con datos de prueba que el modelo no ha visto durante el entrenamiento.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

## 8. Crear el modelo de regresión logística

La regresión logística es un modelo de clasificación.

Aunque su nombre incluye la palabra "regresión", se usa para predecir clases.

En este notebook se usa un `Pipeline` con dos pasos:

1. `StandardScaler`: normaliza las variables para que tengan escalas comparables.
2. `LogisticRegression`: entrena el clasificador.


In [ ]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000)),
    ]
)

model

## 9. Entrenar el modelo

Entrenar significa ajustar el modelo usando los datos de entrenamiento.


In [ ]:
model.fit(X_train, y_train)

## 10. Realizar predicciones

Una vez entrenado el modelo, se usa para predecir las clases de los datos de prueba.


In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Valor real": y_test.values,
    "Predicción": y_pred,
})

results.head(10)

## 11. Calcular la exactitud del modelo

La exactitud indica qué proporción de datos de prueba fueron clasificados correctamente.

Un valor cercano a 1 indica mejor desempeño.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Exactitud del modelo: {accuracy:.4f}")

## 12. Reporte de clasificación

El reporte de clasificación permite revisar el desempeño del modelo para cada clase.

Las métricas principales son:

- `precision`: de las predicciones hechas para una clase, cuántas fueron correctas.
- `recall`: de los casos reales de una clase, cuántos fueron encontrados por el modelo.
- `f1-score`: resumen entre `precision` y `recall`.


In [ ]:
print(classification_report(y_test, y_pred))

## 13. Matriz de confusión

La matriz de confusión muestra los aciertos y errores del modelo.

- La diagonal principal representa las predicciones correctas.
- Los valores fuera de la diagonal representan errores de clasificación.


In [ ]:
labels = ["Normal", "Sospechoso", "Patológico"]

cm = confusion_matrix(y_test, y_pred, labels=labels)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
).plot(values_format="d")

plt.title("Matriz de confusión - Regresión logística")
plt.show()

## 14. Revisar los coeficientes del modelo

La regresión logística calcula coeficientes para las características.

Estos coeficientes ayudan a observar qué variables influyen en la decisión del modelo.

Para facilitar la lectura, se muestra una tabla con los coeficientes asociados a cada clase.


In [ ]:
logistic_model = model.named_steps["classifier"]

coeficients = pd.DataFrame(
    logistic_model.coef_,
    columns=X.columns,
    index=logistic_model.classes_,
)

coeficients

In [ ]:
# Características con mayor peso para cada clase
# Se usa el valor absoluto porque interesa la magnitud del coeficiente.

for class_label in coeficients.index:
    print("Clase:", class_label)
    print(
        coeficients.loc[class_label]
        .abs()
        .sort_values(ascending=False)
        .head(5)
    )

## 15. Conclusión

En este notebook se realizó un flujo básico de clasificación:

1. Leer el archivo CSV con `pandas`.
2. Revisar la estructura de los datos.
3. Separar características y etiqueta.
4. Dividir los datos en entrenamiento y prueba.
5. Entrenar un modelo de regresión logística.
6. Evaluar el modelo con exactitud, reporte de clasificación y matriz de confusión.

Este flujo puede reutilizarse como plantilla para otros problemas de clasificación con bases de datos tabulares.


## 16. Actividad para el estudiante

Responda las siguientes preguntas a partir de los resultados obtenidos:

1. ¿Cuál fue la exactitud del modelo?
2. ¿Cuál clase tuvo mejor desempeño?
3. ¿Cuál clase fue más difícil de clasificar?
4. ¿Qué representa la diagonal principal de la matriz de confusión?
5. ¿Por qué es necesario separar los datos en entrenamiento y prueba?
6. ¿Por qué se usa `StandardScaler` antes de la regresión logística?
7. Cambie el valor de `test_size` a `0.30`. ¿El resultado cambia?
8. Cambie el valor de `random_state`. ¿El resultado cambia?
